# providers.azure

OpenAI chat completions served from an Azure deployment.

In [ ]:
#| default_exp providers.azure

In [ ]:
#| hide
from nbdev.showdoc import *

## The provider

Construction is cheap and side-effect-free — credentials and config are resolved lazily inside `_get_client`. That keeps `from nbdialog.providers.azure import AzureProvider` safe in environments without credentials (offline doc builds, CI without secrets, tests that mock `complete`).

`complete` is a request/response loop, not a single call. When `tools` are passed in, the model can answer directly *or* emit a `tool_calls` array — in which case we dispatch each call against the registered Python function, append the result as a `role: "tool"` message, and re-call. `max_tool_steps` bounds the loop in case the model gets stuck. When no tools are passed, the loop exits on the first response.

Endpoint and deployment have no defaults — set them via the environment or pass them explicitly:

```python
# option 1: env vars (recommended)
# export AZURE_OPENAI_ENDPOINT=https://your-resource.cognitiveservices.azure.com
# export AZURE_OPENAI_DEPLOYMENT=your-deployment-name
# export AZURE_API_KEY=...
set_provider(AzureProvider())

# option 2: explicit
set_provider(AzureProvider(endpoint="https://...", deployment="gpt-5.4"))
```

In [ ]:
#| export
import json, os, time
from openai import AzureOpenAI
from nbdialog.core import Tool, Trace

class AzureProvider:
    "OpenAI chat completions via an Azure deployment, with a tool-call loop and optional Trace recording."
    def __init__(self,
                 deployment: str | None = None,
                 endpoint: str | None = None,
                 api_version: str = "2024-12-01-preview",
                 api_key_env: str = "AZURE_API_KEY",
                 max_completion_tokens: int = 16384,
                 max_tool_steps: int = 8):
        self.deployment = deployment or os.environ.get("AZURE_OPENAI_DEPLOYMENT")
        self.endpoint = endpoint or os.environ.get("AZURE_OPENAI_ENDPOINT")
        self.api_version, self.api_key_env = api_version, api_key_env
        self.max_completion_tokens = max_completion_tokens
        self.max_tool_steps = max_tool_steps
        self._client = None

    def _get_client(self):
        if self._client is None:
            if not self.endpoint:
                raise RuntimeError("AzureProvider: no endpoint. Set $AZURE_OPENAI_ENDPOINT or pass endpoint=...")
            if not self.deployment:
                raise RuntimeError("AzureProvider: no deployment. Set $AZURE_OPENAI_DEPLOYMENT or pass deployment=...")
            self._client = AzureOpenAI(api_key=os.environ[self.api_key_env],
                                       azure_endpoint=self.endpoint,
                                       api_version=self.api_version)
        return self._client

    def complete(self, messages: list[dict], tools: list[Tool] = None,
                 trace: Trace = None) -> str:
        tools = tools or []
        schemas = [t.schema for t in tools]
        dispatch = {t.schema["function"]["name"]: t.fn for t in tools}
        msgs = list(messages)
        for _ in range(self.max_tool_steps):
            kw = {"tools": schemas} if schemas else {}
            t0 = time.perf_counter()
            resp = self._get_client().chat.completions.create(
                model=self.deployment, messages=msgs,
                max_completion_tokens=self.max_completion_tokens, **kw)
            dt = time.perf_counter() - t0
            m = resp.choices[0].message
            if trace is not None:
                tc_for_trace = [{"name": tc.function.name,
                                 "args": json.loads(tc.function.arguments)}
                                for tc in (m.tool_calls or [])]
                usage = resp.usage.model_dump() if resp.usage else None
                trace.add_model_turn(m.content, tc_for_trace, usage, dt)
            if not m.tool_calls: return m.content
            msgs.append(m.model_dump(exclude_none=True))
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                t0 = time.perf_counter()
                out = dispatch[tc.function.name](**args)
                dt = time.perf_counter() - t0
                if trace is not None:
                    trace.add_tool_turn(tc.function.name, args, str(out), dt)
                msgs.append({"role": "tool", "tool_call_id": tc.id,
                             "content": str(out)})
        raise RuntimeError(f"Tool loop did not converge in {self.max_tool_steps} steps")

### Tests

Exercise the tool-call loop against a scripted fake client — no network, no credentials. Run with `nbdev_test`.

In [ ]:
# A scripted stand-in for openai.AzureOpenAI: returns a queued list of responses
# and records the kwargs of each `.chat.completions.create(...)` call.
from types import SimpleNamespace

def _tc(call_id, name, **args):
    "Build a fake tool_call matching what the openai SDK produces."
    fn = SimpleNamespace(name=name, arguments=json.dumps(args))
    return SimpleNamespace(id=call_id, type="function", function=fn)

def _msg(content=None, tool_calls=None):
    "Build a fake assistant message; model_dump mirrors the shape the loop appends to msgs."
    def dump(exclude_none=False):
        d = {"role": "assistant", "content": content}
        if tool_calls:
            d["tool_calls"] = [{"id": tc.id, "type": "function",
                                "function": {"name": tc.function.name,
                                             "arguments": tc.function.arguments}}
                               for tc in tool_calls]
        return {k:v for k,v in d.items() if v is not None} if exclude_none else d
    return SimpleNamespace(content=content, tool_calls=tool_calls, model_dump=dump)

def _resp(msg, total_tokens=10):
    usage = SimpleNamespace(model_dump=lambda: {"total_tokens": total_tokens})
    return SimpleNamespace(choices=[SimpleNamespace(message=msg)], usage=usage)

class _FakeClient:
    "Drop-in for openai.AzureOpenAI exposing .chat.completions.create(...)."
    def __init__(self, responses):
        self._responses = list(responses)
        self.calls = []
    @property
    def chat(self):        return self
    @property
    def completions(self): return self
    def create(self, **kw):
        self.calls.append(kw)
        return self._responses.pop(0)

def _provider(client, **kw):
    "AzureProvider wired to a fake client; explicit endpoint/deployment skip env lookup."
    prov = AzureProvider(endpoint="http://fake", deployment="fake", **kw)
    prov._client = client
    return prov

In [ ]:
# No tools registered: single round-trip, returns content, no `tools` kwarg sent.
client = _FakeClient([_resp(_msg(content="pong"))])
prov = _provider(client)
assert prov.complete([{"role":"user","content":"ping"}]) == "pong"
assert len(client.calls) == 1
assert "tools" not in client.calls[0]

In [ ]:
# Single tool call: dispatch runs, result is appended as role=tool, model resolves.
def _add(x, y): return x + y
add = Tool(
    schema={"type":"function","function":{"name":"add","parameters":{}}},
    fn=_add,
)
client = _FakeClient([
    _resp(_msg(tool_calls=[_tc("c1", "add", x=2, y=3)])),
    _resp(_msg(content="5")),
])
prov = _provider(client)
assert prov.complete([{"role":"user","content":"2+3?"}], tools=[add]) == "5"
assert len(client.calls) == 2
# the second call carries the tool result as the last message
tail = client.calls[1]["messages"][-1]
assert tail == {"role":"tool","tool_call_id":"c1","content":"5"}
# and the assistant message dump (with tool_calls) sits just before it
assert client.calls[1]["messages"][-2]["role"] == "assistant"
assert client.calls[1]["messages"][-2]["tool_calls"][0]["function"]["name"] == "add"

In [ ]:
# A model that never stops calling tools should raise after max_tool_steps.
def _noop(**_): return "ok"
noop = Tool(schema={"type":"function","function":{"name":"noop","parameters":{}}}, fn=_noop)
client = _FakeClient([_resp(_msg(tool_calls=[_tc(f"c{i}", "noop")])) for i in range(10)])
prov = _provider(client, max_tool_steps=3)
try:
    prov.complete([{"role":"user","content":"go"}], tools=[noop])
    assert False, "expected RuntimeError"
except RuntimeError as e:
    assert "did not converge" in str(e)
assert len(client.calls) == 3

In [ ]:
# Trace records every model turn and every tool turn, in order, with timings.
def _double(n): return n * 2
double = Tool(schema={"type":"function","function":{"name":"double","parameters":{}}}, fn=_double)
client = _FakeClient([
    _resp(_msg(tool_calls=[_tc("c1", "double", n=21)]), total_tokens=42),
    _resp(_msg(content="42"),                          total_tokens=7),
])
prov = _provider(client)
tr = Trace()
assert prov.complete([{"role":"user","content":"2*21?"}], tools=[double], trace=tr) == "42"
kinds = [t["kind"] for t in tr.turns]
assert kinds == ["model", "tool", "model"], kinds
assert tr.turns[1]["name"] == "double" and tr.turns[1]["result"] == "42"
assert tr.turns[0]["tool_calls"][0]["name"] == "double"
assert tr._tokens == 49  # 42 + 7

Smoke test — only runs when credentials are present, so the doc build stays green without them.

In [ ]:
if all(os.environ.get(k) for k in ("AZURE_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_DEPLOYMENT")):
    print(AzureProvider().complete([{"role":"user","content":"ping"}]))

pong


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()